# Lab 30: Sensor Data Streaming, Anomaly Detection and Real-Time Dashboard

**Duration:** 3 hours. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
In Labs 28 and 29 the AI looked at **pictures**. Most IoT data are not pictures but **numbers over time**: a temperature, a light level, a vibration. In this lab the ESP32 board is the **sensor node**: it measures and sends numbers all day. The Jetson is the **edge AI**: it stores the numbers, learns what is **normal**, finds **unusual** values (anomalies), warns early with a **trend**, switches the alert on the board, and shows everything on a **live dashboard**. No cloud is needed: the data and the decisions stay in your room.

**By the end of this lab you can:**
1. Collect sensor data from MQTT and store it in a database (SQLite) and in Excel.
2. Query the data with SQL.
3. Find anomalies with a moving average and a z-score, and judge it with false alarms and misses.
4. Predict a trend with a straight line and warn before a limit is reached.
5. Show a live dashboard and detect when a sensor goes silent.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. Press **Ctrl + S** often.

> **Board.** The lab uses the **virtual ESP32** (Lab 28). It sends `esp/sensor/light` and `esp/sensor/temp` every 2 seconds and can make a **made-up disturbance** (`board.disturb(...)`) to test your alarm. With the real board the same topics arrive: the light sensor and the chip temperature. Nothing in the Jetson program changes.

## 2. Background

**Time series.** A list of (time, value) pairs. Sensor data are time series. Three things matter: the **level** (the average), the **spread** (how much it jumps around when all is normal) and the **change** (a trend or a spike).

**Storing data.** A CSV file is simple but slow to search. A **database** answers questions. **SQLite** is a database in one file, built into Python (`sqlite3`). A table `readings(time, sensor, value)` with `INSERT` to store and `SELECT` to ask, for example the average of each minute:

`SELECT strftime('%H:%M', time) AS minute, AVG(value) FROM readings WHERE sensor = 'temp' GROUP BY minute`

**What is an anomaly?** A value that is **unusual compared with the recent past**. A fixed limit (alarm above 40 degrees) is simple, but it does not know what is normal for this room. A **z-score** does:

`z = (value - mean of the window) / standard deviation of the window`

The **window** is the last `WINDOW` values. A z-score of 0 is exactly average. About 99.7 % of normal values have |z| below 3. So `|z| > 3` is a **rare** value: an anomaly.

| Setting | If you raise it | If you lower it |
|---|---|---|
| `Z_LIMIT` | Fewer false alarms, but small changes are missed | More alarms, more false alarms |
| `WINDOW` | A slow, stable normal, but slow to adapt to real changes | Adapts fast, but a long disturbance becomes "normal" |
| `CONFIRM` (values in a row) | Fewer false alarms from one noisy value | The alarm is faster |
| `min_spread` (the noise floor of the sensor) | Small jitter never counts as an anomaly | Tiny changes of a very steady sensor are reported |

The **window** can be unlucky: if its first values are very steady, it underestimates the normal jitter and a perfectly normal value looks rare. The **noise floor** `min_spread` (the smallest jitter that is normal for this sensor, for example 0.5 degrees for the chip temperature) protects against this.

**Trend and early warning.** A straight line through the last values (`np.polyfit(time, value, 1)`) gives the **slope**: how fast the value rises per second. With the slope you can predict when the line reaches a limit: `time to limit = (limit - now) / slope`. A warning "the temperature reaches 40 degrees in 6 minutes" is more useful than an alarm when it is already 40.

**Edge AI.** All of this runs on the Jetson, next to the sensor. The data do not leave the building, the reaction is immediate, and it still works when the internet is down.

**A silent sensor is also a problem.** If no data arrive for a while, the sensor, the board or the network is broken. The dashboard must show **no data** and not the last old value. This is a **stale data** check.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(30)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Collect and Store the Data (40 min)

### Step 1: The Board and the Collector
The cell starts the virtual board and the link (as in Lab 28). A second MQTT client, the **collector**, subscribes to `esp/sensor/#` and writes **every reading** into a SQLite database `sensors.db`. A lock keeps the database safe because the messages arrive in another thread.

In [ ]:
# ---------------- Settings you can change ----------------
WINDOW = 30           # how many values make the window for "normal"
Z_LIMIT = 3.0         # a value more than this many standard deviations from the average is unusual
CONFIRM = 2           # how many unusual values in a row make an anomaly
STALE = 10.0          # seconds without data before a sensor counts as silent
# ----------------------------------------------------------
import csv
import os
import sqlite3
import threading
import time
from collections import deque
from datetime import datetime

import cv2
import numpy as np

import labview

board = labview.VirtualEsp32()
esp = labview.Esp32()

db = sqlite3.connect("sensors.db", check_same_thread=False)
db_lock = threading.Lock()
db.execute("CREATE TABLE IF NOT EXISTS readings (time REAL, sensor TEXT, value REAL)")
db.commit()
latest = {}                      # sensor -> (time, value)


def on_reading(topic, text):
    """Called for every message of esp/sensor/#: keep the newest value and store it in the database."""
    try:
        value = float(text)
    except ValueError:
        return
    sensor, now = topic.split("/")[-1], time.time()
    latest[sensor] = (now, value)
    with db_lock:
        db.execute("INSERT INTO readings VALUES (?, ?, ?)", (now, sensor, value))
        db.commit()


collector = labview.mqtt(on_reading, subscribe=["esp/sensor/#"])
time.sleep(5)
print("Latest values:", {k: round(v[1], 1) for k, v in latest.items()})

### Step 2: Ask the Database
Let the data collect for about a minute, then use SQL. `run_sql` runs a question and prints the rows.

In [ ]:
def run_sql(query, args=()):
    with db_lock:
        return db.execute(query, args).fetchall()


time.sleep(40)
print("Readings in the table:", run_sql("SELECT COUNT(*) FROM readings")[0][0])
print()
print(f"{'sensor':<8}{'count':>7}{'average':>10}{'lowest':>9}{'highest':>9}")
for sensor, count, avg, low, high in run_sql("SELECT sensor, COUNT(*), AVG(value), MIN(value), MAX(value) FROM readings GROUP BY sensor"):
    print(f"{sensor:<8}{count:>7}{avg:>10.1f}{low:>9.1f}{high:>9.1f}")
print()
print("The last 5 temperature readings:")
for t, value in run_sql("SELECT time, value FROM readings WHERE sensor = 'temp' ORDER BY time DESC LIMIT 5"):
    print("  ", datetime.fromtimestamp(t).strftime("%H:%M:%S"), round(value, 1))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- How many readings did you collect and from which sensors?:
- Average, lowest and highest of the temperature and of the light:
- Why does each sensor have about the same number of readings? How often does the board send?:


Write your **own** SQL question. For example: how many light readings were above 2000? Change the `WHERE` part.

In [ ]:
print(run_sql("SELECT COUNT(*) FROM readings WHERE sensor = 'light' AND value > 2000"))

### Step 3: Export to Excel
Excel is how many people look at data. The cell writes the readings of the last 5 minutes into `sensor_data.csv` and `sensor_data.xlsx`.

In [ ]:
def export_excel(minutes=5):
    rows = run_sql("SELECT time, sensor, value FROM readings WHERE time > ? ORDER BY time", (time.time() - 60 * minutes,))
    with open("sensor_data.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["time", "sensor", "value"])
        for t, sensor, value in rows:
            writer.writerow([datetime.fromtimestamp(t).strftime("%Y-%m-%d %H:%M:%S"), sensor, round(value, 2)])
    try:
        from openpyxl import Workbook
        from openpyxl.styles import Font
        book = Workbook()
        sheet = book.active
        sheet.title = "Sensors"
        sheet.append(["time", "sensor", "value"])
        for t, sensor, value in rows:
            sheet.append([datetime.fromtimestamp(t).strftime("%Y-%m-%d %H:%M:%S"), sensor, round(value, 2)])
        for cell in sheet[1]:
            cell.font = Font(bold=True)
        sheet.column_dimensions["A"].width = 20
        book.save("sensor_data.xlsx")
    except ImportError:
        print("sensor_data.xlsx needs openpyxl. sensor_data.csv works in Excel too.")
    print(len(rows), "readings exported. Download sensor_data.xlsx from the file browser to open it on your laptop.")


export_excel()

---
# Part B: Find the Unusual (50 min)

### Step 1: The Anomaly Detector
`Anomaly` keeps the last `WINDOW` values. For a new value it calculates the z-score **against the window** (before the new value is added). It needs at least 20 values before it judges (`warm-up`). A value is **unusual** when `|z| > Z_LIMIT`. It is an **anomaly** after `CONFIRM` unusual values in a row. **Unusual values are not added to the window**, so a disturbance does not teach the detector that it is normal. The cell tests it on made-up data.

In [ ]:
class Anomaly:
    def __init__(self, window=None, z_limit=None, confirm=None, min_spread=0.0):
        self.min_spread = min_spread                               # the smallest normal jitter of the sensor (its noise floor)
        self.window = deque(maxlen=window or WINDOW)
        self.z_limit = z_limit or Z_LIMIT
        self.confirm = confirm or CONFIRM
        self.streak = 0

    def update(self, value):
        """Judge one value. Returns (z-score, anomaly yes or no)."""
        if len(self.window) < 20:
            self.window.append(value)
            return 0.0, False                                       # warm-up: not enough data yet
        mean = float(np.mean(self.window))
        spread = max(float(np.std(self.window)), self.min_spread, 1e-6)      # a short window can underestimate the jitter
        z = (value - mean) / spread
        if abs(z) > self.z_limit:
            self.streak += 1                                        # unusual: do NOT learn it
        else:
            self.streak = 0
            self.window.append(value)
        return z, self.streak >= self.confirm


rng = np.random.default_rng(1)
values = list(25 + rng.normal(0, 0.3, 40)) + [31, 31.5, 32] + list(25 + rng.normal(0, 0.3, 10))     # 40 normal, 3 high, 10 normal
detector = Anomaly(min_spread=0.3)                 # the jitter of this made-up sensor is about 0.3
flags = [detector.update(v) for v in values]
print("Values 40 to 43:", [(round(v, 1), round(z, 1), a) for v, (z, a) in zip(values[40:44], flags[40:44])])
print("Anomalies at indexes:", [i for i, (z, a) in enumerate(flags) if a], "(expected 41 and 42: the second and third high value)")
print("Value 43 (normal again):", round(flags[43][0], 1), flags[43][1])

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- At which indexes was an anomaly reported? Why not at index 40?:
- Why is an unusual value not added to the window?:
- What happens if `CONFIRM = 1`? What if `CONFIRM = 5`?:
- Change `min_spread=0.3` to `0.0` and run again. What happens and why? (Try with `Anomaly(min_spread=0.0)` and a window of 20 normal values.):


### Step 2: Compare with a Fixed Limit
A fixed limit (`temp > 28`) is simple. Compare it with the z-score on data where the **room is slowly warming**: the normal level changes during the day. Which one gives false alarms?

In [ ]:
rng = np.random.default_rng(2)
day = 22 + np.linspace(0, 8, 200) + rng.normal(0, 0.3, 200)           # a room that warms up from 22 to 30 degrees
day[150:153] += 5                                                      # a real disturbance (3 values)

fixed_alarms = [i for i, v in enumerate(day) if v > 28]
zdet = Anomaly(min_spread=0.3)
z_alarms = [i for i, v in enumerate(day) if zdet.update(v)[1]]
print("Fixed limit (above 28):  alarms =", len(fixed_alarms), "| first at", fixed_alarms[:1], "| catches the real disturbance (150 to 152):", any(150 <= i <= 152 for i in fixed_alarms))
print("z-score:                 alarms =", len(z_alarms), "| indexes", z_alarms[:8])

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- How many alarms did each method give?:
- Which method gives false alarms on a slowly warming room? Why?:
- Which method catches the real disturbance? Is it caught by both?:


### Step 3: Trend and Early Warning
`predict_time_to(limit, points)` fits a straight line to the last points and returns the **seconds until the value reaches the limit**. It returns `None` if the value is not rising towards it. The cell tests it on a made-up rise of 0.1 degree per second.

In [ ]:
def predict_time_to(limit, points):
    """points: a list of (time in seconds, value). Seconds until the fitted line reaches `limit`, or None."""
    if len(points) < 5:
        return None
    t = np.array([p[0] for p in points], dtype=float)
    v = np.array([p[1] for p in points], dtype=float)
    slope, intercept = np.polyfit(t - t[-1], v, 1)                 # time counted from the newest point
    now_value = intercept
    if abs(slope) < 1e-9 or (limit - now_value) / slope < 0:
        return None                                                 # not moving towards the limit
    return (limit - now_value) / slope


rising = [(t, 30 + 0.1 * t + float(np.random.default_rng(3).normal(0, 0.05))) for t in range(0, 30, 2)]     # the last value is about 32.8
print("Rising 0.1 per second, limit 40:", round(predict_time_to(40, rising)), "s (expected about 72)")
print("Falling, limit 40:              ", predict_time_to(40, [(t, 30 - 0.1 * t) for t in range(0, 30, 2)]), "(expected None)")
print("Flat, limit 40:                 ", predict_time_to(40, [(t, 30.0) for t in range(0, 30, 2)]), "(expected None)")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Were the three results as expected?:
- Why is a warning 'the limit is reached in 70 seconds' more useful than an alarm at the limit?:
- What can go wrong with a straight-line prediction? (Think of a value that does not rise for ever.):


---
# Part C: The Live Dashboard (50 min)

### Step 1: Draw a Chart
`draw_chart` draws a line chart of the last `seconds` of a sensor with OpenCV (Lab 6): the grid, the line, the red dots of the anomalies, the current value and a title. The cell tests it with the data of the day above.

In [ ]:
from IPython.display import Image, display


def draw_chart(points, anomalies, title, width=620, height=190, seconds=120, unit=""):
    """points: a list of (time, value). anomalies: a list of times. Returns a picture (OpenCV)."""
    picture = np.full((height, width, 3), 250, np.uint8)
    cv2.putText(picture, title, (10, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (60, 60, 60), 1)
    if len(points) < 2:
        cv2.putText(picture, "no data yet", (width // 2 - 50, height // 2), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 200), 1)
        return picture
    now = points[-1][0]
    shown = [p for p in points if p[0] >= now - seconds]
    low, high = min(v for _, v in shown), max(v for _, v in shown)
    pad = max((high - low) * 0.15, 0.5)
    low, high = low - pad, high + pad
    left, top, bottom = 50, 30, height - 22
    to_x = lambda t: int(left + (t - (now - seconds)) / seconds * (width - left - 10))
    to_y = lambda v: int(bottom - (v - low) / (high - low) * (bottom - top))
    for fraction in (0, 0.5, 1):
        y = int(bottom - fraction * (bottom - top))
        cv2.line(picture, (left, y), (width - 10, y), (225, 225, 225), 1)
        cv2.putText(picture, f"{low + fraction * (high - low):.1f}", (4, y + 4), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (110, 110, 110), 1)
    for a, b in zip(shown, shown[1:]):
        cv2.line(picture, (to_x(a[0]), to_y(a[1])), (to_x(b[0]), to_y(b[1])), (160, 90, 0), 2)
    for t in anomalies:
        near = min(shown, key=lambda p: abs(p[0] - t))
        if abs(near[0] - t) < 3 and now - seconds <= t:
            cv2.circle(picture, (to_x(near[0]), to_y(near[1])), 5, (0, 0, 220), -1)
    cv2.putText(picture, f"{shown[-1][1]:.1f}{unit}", (width - 120, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (160, 90, 0), 2)
    cv2.putText(picture, f"last {seconds} s", (width - 90, height - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (110, 110, 110), 1)
    return picture


test = [(float(i), float(v)) for i, v in enumerate(day)]
picture = draw_chart(test, [150.0, 151.0, 152.0], "Test: a room that warms up", seconds=200, unit=" C")
display(Image(data=cv2.imencode(".png", picture)[1].tobytes()))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Where are the red dots on the test chart?:
- What would you add to the chart to make it clearer (a limit line, colours, a title)?:


### Step 2: The Dashboard
The `Dashboard` runs in a thread. Every second it:
1. takes the newest value of each sensor from `latest`, judges it with an `Anomaly` detector (one detector for each sensor),
2. checks for **stale data** (no value for `STALE` seconds, the sensor is silent),
3. switches the board **alert on** when any sensor has an anomaly (and **off** when all are normal for 10 seconds). It sends a command only when the state changes (Lab 28),
4. publishes a JSON summary to the topic `dash/state` (retained) for your phone,
5. draws the charts and the status tiles.

Run the cell. Then make a disturbance in **another cell**: `board.disturb("temp", 12, 8)` makes the temperature 12 degrees higher for 8 seconds. Watch the dots, the tile and the board. Press **Stop** when you are done.

In [ ]:
import json

import ipywidgets as widgets

# min_spread: the smallest normal jitter of the sensor. limit: the value for the early warning.
SENSORS = {"temp": {"title": "Temperature", "unit": " C", "limit": 40.0, "min_spread": 0.5},
           "light": {"title": "Light", "unit": "", "limit": None, "min_spread": 60.0}}


class Dashboard:
    def __init__(self):
        self.detectors = {name: Anomaly(min_spread=spec["min_spread"]) for name, spec in SENSORS.items()}
        self.history = {name: [] for name in SENSORS}
        self.normal = {name: [] for name in SENSORS}               # only the normal values: the trend must not learn a spike
        self.marks = {name: [] for name in SENSORS}
        self.last_seen = {}
        self.alert = False
        self.normal_since = time.time()
        self.stop_flag = threading.Event()
        self.picture = widgets.Image(format="png")
        self.status = widgets.Label(value="Starting ...")
        self.button = widgets.Button(description="Stop", button_style="danger")
        self.button.on_click(lambda _: self.stop())
        display(widgets.VBox([self.button, self.status, self.picture]))
        threading.Thread(target=self._run, daemon=True).start()

    def step(self, now):
        """One second of work. Returns a list of text lines for the status."""
        lines, any_anomaly = [], False
        for name, spec in SENSORS.items():
            seen = latest.get(name)
            if seen is None:
                lines.append(f"{spec['title']}: no data yet")
                continue
            t, value = seen
            if now - t > STALE:
                lines.append(f"{spec['title']}: NO DATA for {now - t:.0f} s (sensor silent)")
                continue
            if self.last_seen.get(name) != t:                       # a new reading: judge it once
                self.last_seen[name] = t
                self.history[name].append((t, value))
                self.history[name] = self.history[name][-400:]
                z, anomaly = self.detectors[name].update(value)
                if anomaly:
                    self.marks[name].append(t)
                if self.detectors[name].streak == 0:
                    self.normal[name] = (self.normal[name] + [(t, value)])[-60:]
            anomaly_now = self.detectors[name].streak >= CONFIRM
            any_anomaly = any_anomaly or anomaly_now
            text = f"{spec['title']}: {value:.1f}{spec['unit']}" + ("   ANOMALY" if anomaly_now else "   normal")
            to_limit = predict_time_to(spec["limit"], self.normal[name][-15:]) if spec["limit"] and not anomaly_now else None
            if to_limit is not None and to_limit < 600:
                text += f"   (reaches {spec['limit']:.0f} in {to_limit:.0f} s)"
            lines.append(text)
        if any_anomaly:
            self.normal_since = now
        want = any_anomaly or (self.alert and now - self.normal_since < 10)
        if want != self.alert:                                       # a command only when the state changes
            self.alert = want
            if esp.online():
                esp.send("alert", "on" if want else "off")
            esp.client.publish("dash/state", json.dumps({"alert": want, "values": {k: v[1] for k, v in latest.items()}}), retain=True)
        return lines

    def _run(self):
        while not self.stop_flag.is_set():
            lines = self.step(time.time())
            self.status.value = ("ALERT ON | " if self.alert else "all normal | ") + " | ".join(lines)
            charts = [draw_chart(self.history[n], self.marks[n], SENSORS[n]["title"], unit=SENSORS[n]["unit"]) for n in SENSORS]
            self.picture.value = cv2.imencode(".png", np.vstack(charts))[1].tobytes()
            self.stop_flag.wait(1.0)

    def stop(self):
        self.stop_flag.set()
        self.status.value += "   [stopped]"


dashboard = Dashboard()

Make a disturbance (run this cell **while** the dashboard runs, after it has run for at least 30 seconds so the detector has learned what is normal):

In [ ]:
board.disturb("temp", 12, 8)           # the temperature is 12 degrees higher for 8 seconds
print("Disturbance started. Watch the dashboard and the board:", board.describe())

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- How long after the disturbance did the dashboard show ANOMALY? Why not at once?:
- What did the board do? (look at `board.describe()` in a new cell):
- How long after the disturbance ended did the alert go off? Why that long?:


Try also `board.disturb("light", 1500, 10)`, and **a sensor that goes silent**: stop the virtual board with `board.stop()` and watch the dashboard say `NO DATA`. Start it again with `board = labview.VirtualEsp32()`.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- What did the dashboard show for the light disturbance?:
- What did it show when the board was stopped? After how many seconds?:
- What would happen if the dashboard kept showing the last old value?:


---
# Part D: Make It Your Own, Test and Present (40 min)

### Step 1: Make It Your Own
Add **at least one improvement** from the list, or one of your own. Change the code in the cells above, run that cell again, then run the dashboard again.

- Change the **sensors**: with the real board add a DHT22 (temperature and humidity), a soil sensor or a sound level to `esp32/main.py` (publish `esp/sensor/<name>`) and add the name to `SENSORS`.
- Tune **`Z_LIMIT`, `WINDOW` and `CONFIRM`** and write a table of false alarms and missed anomalies for three settings (use `board.disturb` 10 times and count).
- Add a **fixed limit** as a second alarm (the temperature above `limit` is always an alarm, even if it rose slowly).
- Add a **daily report**: a cell that writes the minimum, maximum and average of each hour with SQL.
- Publish the values to your **phone** (`dash/state`) and show them in the IoT MQTT Panel app; send `alert off` from the phone.
- Add the **picture of Lab 28**: the camera counts people and the dashboard shows the count next to the temperature.

> **Tip.** Change one thing at a time. To keep a copy of a cell before a big change, copy the cell (select it, press **C**, then **V**) and change the copy.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Improvement you chose:
- Which lines did you change?:
- Did it work the first time? What went wrong?:


### Step 2: Test
Test each situation and write what happened.

| Test | Situation | Expected | What happened |
|---|---|---|---|
| 1 | Run the dashboard for 2 minutes with no disturbance | No anomaly (zero false alarms) | |
| 2 | `board.disturb("temp", 12, 8)` | Anomaly, alert on, then off | |
| 3 | `board.disturb("temp", 3, 8)` (a small change) | Anomaly or not? Judge it | |
| 4 | `board.disturb("light", 1500, 10)` | Anomaly in the light only | |
| 5 | Three disturbances one after the other | Does the detector keep working? | |
| 6 | `board.stop()` for 15 seconds | `NO DATA` | |
| 7 | The board comes back | Data and charts continue | |
| 8 | A slow disturbance: `board.disturb("temp", 6, 60)` | Is a long disturbance still reported after 60 seconds? | |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Test 1 to 4:
- Test 5 to 7:
- Test 8: what happens to a very long disturbance? Which setting controls it?:
- Count: how many false alarms and how many missed anomalies did you see in all tests?:


### Step 3: Report and Present
Fill in the report. Then give a **3-minute demonstration**: the data in the database and an SQL question, the z-score detector with a disturbance, the trend warning, the live dashboard with the board alert, and the silent sensor.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The sensors used and how often they send:
- Settings: `WINDOW`, `Z_LIMIT`, `CONFIRM`, and why:
- Power mode used (run  !sudo nvpmodel -q  in a cell, after  labview.sudo_login()  ):
- Your improvement:
- One problem you found and how you fixed it:
- One idea to make it better in future:


> **Where next.** In Lab 31 you combine everything into your own **capstone project**: an AIoT application of your choice, with the Jetson as the brain and the ESP32 as the body.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The readings are stored in the database and an SQL question is answered | |
| 2 | The Excel export is shown | |
| 3 | The anomaly test finds the made-up disturbance and the settings are explained | |
| 4 | The comparison of the fixed limit and the z-score is explained | |
| 5 | The trend prediction gives the expected numbers | |
| 6 | The live dashboard shows two charts and the status | |
| 7 | A disturbance gives an anomaly, an alert on the board, and then clears | |
| 8 | The silent sensor is shown as `NO DATA`; at least one improvement works | |
| 9 | The report is complete and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- 1. What is a time series? Give one example from your own life.:
- 2. Why use a database and not only a CSV file? Write one SQL question.:
- 3. What is a z-score and what does a z-score of 4 mean?:
- 4. Why is a z-score better than a fixed limit for a room that warms up during the day?:
- 5. Why are unusual values not added to the window?:
- 6. What does the `CONFIRM` setting prevent? What is its cost?:
- 7. How does the straight-line trend give an early warning?:
- 8. What is stale data? Why must a dashboard check for it?:
- 9. Why is it a benefit to run the AI on the Jetson, next to the sensor? (a) It still works without the internet and reacts at once (b) It is cheaper than a cloud (c) The data stay in the room (d) All of these:


## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `latest` is empty | The board is not running or not on your broker | Run the first cell again. Check `esp.online()`. |
| `database is locked` | Two programs write at once | Use the `run_sql` function (it has a lock). Close other notebooks that use `sensors.db`. |
| The SQL gives no rows | The sensor name is different, or the time filter is too strict | `SELECT DISTINCT sensor FROM readings` shows the names |
| The detector says anomaly all the time | The window is too small, the value jumps a lot, or `Z_LIMIT` is too low | Raise `WINDOW` and `Z_LIMIT`. Check the chart. |
| The detector never finds anything | `Z_LIMIT` is too high or `CONFIRM` too big, or the disturbance is small | Lower them. Try `board.disturb("temp", 12, 8)`. |
| The disturbance is found only at first | A long disturbance is not learned (unusual values are not added) | This is intended: the alarm stays on. Lower `CONFIRM` is not needed. |
| The chart shows `no data yet` | Not enough readings yet | Wait 10 seconds |
| `ImportError ... ipywidgets` | The widgets are not installed | Ask your teacher (they come with Lab 6) |
| The dashboard does not stop | The Stop button is above the charts | Press it, or run `dashboard.stop()` in a cell |
| The alert on the board never switches off | The anomaly continues, or `esp.online()` was false | Wait 10 seconds after the last anomaly. Send `esp.send("alert", "off")` by hand. |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(30)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- SQLite: https://www.sqlite.org/
- Python documentation, sqlite3: https://docs.python.org/3/library/sqlite3.html
- Standard score (z-score): https://en.wikipedia.org/wiki/Standard_score
- Anomaly detection: https://en.wikipedia.org/wiki/Anomaly_detection
- NumPy, polyfit: https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html